In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import VotingRegressor,RandomForestRegressor, BaggingRegressor,StackingRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_transformer,make_column_selector
from sklearn.impute import SimpleImputer
os.chdir('/home/pgcp-ai/MachineLearning/Datasets/BigMarketSale/')

/home/pgcp-ai/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
train = pd.read_csv("train_v9rqX0R.csv")
test = pd.read_csv("test_AbJTz2l.csv")

In [3]:
train

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.300,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.920,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.500,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.200,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.930,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052
...,...,...,...,...,...,...,...,...,...,...,...,...
8518,FDF22,6.865,Low Fat,0.056783,Snack Foods,214.5218,OUT013,1987,High,Tier 3,Supermarket Type1,2778.3834
8519,FDS36,8.380,Regular,0.046982,Baking Goods,108.1570,OUT045,2002,NaN,Tier 2,Supermarket Type1,549.2850
8520,NCJ29,10.600,Low Fat,0.035186,Health and Hygiene,85.1224,OUT035,2004,Small,Tier 2,Supermarket Type1,1193.1136
8521,FDN46,7.210,Regular,0.145221,Snack Foods,103.1332,OUT018,2009,Medium,Tier 3,Supermarket Type2,1845.5976


In [4]:
train['Item_Weight'].isnull().sum()

1463

In [5]:
train.groupby('Item_Identifier')['Item_Weight'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

Item_Identifier
DRA12    11.600
DRA24    19.350
DRA59     8.270
DRB01     7.390
DRB13     6.115
          ...  
NCZ30     6.590
NCZ41    19.850
NCZ42    10.500
NCZ53     9.600
NCZ54    14.650
Name: Item_Weight, Length: 1559, dtype: float64

In [6]:
train['Item_Weight'] = train.groupby('Item_Identifier')['Item_Weight'].ffill()

In [7]:
train.groupby('Item_Identifier')['Item_Weight'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

Item_Identifier
DRA12    11.600
DRA24    19.350
DRA59     8.270
DRB01     7.390
DRB13     6.115
          ...  
NCZ30     6.590
NCZ41    19.850
NCZ42    10.500
NCZ53     9.600
NCZ54    14.650
Name: Item_Weight, Length: 1559, dtype: float64

In [8]:
train['Item_Weight'].isnull().sum()

272

In [9]:
train['Item_Identifier'].isnull().sum()

0

In [10]:
train.isnull().sum()

Item_Identifier                 0
Item_Weight                   272
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [11]:
train['Item_Fat_Content'].unique()

array(['Low Fat', 'Regular', 'low fat', 'LF', 'reg'], dtype=object)

In [12]:
items_trn = train[['Item_Identifier','Item_Weight']].sort_values(by='Item_Identifier')
items_trn = items_trn[items_trn['Item_Weight'].notna()]

In [13]:
items_tst = test[['Item_Identifier','Item_Weight']].sort_values(by='Item_Identifier')
items_tst = items_tst[items_tst['Item_Weight'].notna()]

In [14]:
items = pd.concat([items_trn,items_tst])
items = items.drop_duplicates()
train.drop('Item_Weight',axis=1,inplace=True)

In [15]:
train_weight = train.merge(items,how='inner',on='Item_Identifier')
train_weight.isnull().sum()

Item_Identifier                 0
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
Item_Weight                     0
dtype: int64

In [16]:
train_weight

,Item_Identifier,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Item_Weight
0,FDA15,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,9.300
1,FDA15,Low Fat,0.016055,Dairy,250.2092,OUT045,2002,NaN,Tier 2,Supermarket Type1,5976.2208,9.300
2,FDA15,Low Fat,0.016019,Dairy,248.5092,OUT035,2004,Small,Tier 2,Supermarket Type1,6474.2392,9.300
3,FDA15,Low Fat,0.016088,Dairy,249.6092,OUT018,2009,Medium,Tier 3,Supermarket Type2,5976.2208,9.300
4,FDA15,Low Fat,0.026818,Dairy,248.9092,OUT010,1998,NaN,Tier 3,Grocery Store,498.0184,9.300
...,...,...,...,...,...,...,...,...,...,...,...,...
8518,NCF55,LF,0.021666,Household,33.3874,OUT046,1997,Small,Tier 1,Supermarket Type1,1235.0590,6.675
8519,NCW30,Low Fat,0.011072,Household,257.8962,OUT017,2007,NaN,Tier 2,Supermarket Type1,4402.9354,5.210
8520,NCW30,Low Fat,0.011008,Household,259.5962,OUT035,2004,Small,Tier 2,Supermarket Type1,2848.9582,5.210
8521,NCW05,Low Fat,0.148303,Health and Hygiene,108.3938,OUT049,1999,Medium,Tier 1,Supermarket Type1,2787.0388,20.250


In [17]:
train_weight['Item_Fat_Content'].replace({
    'low fat': 'Low Fat', 
    'LF': 'Low Fat',
    'reg': 'Regular'
}, inplace=True)

In [18]:
train_weight['Item_Fat_Content'].unique()

array(['Low Fat', 'Regular'], dtype=object)

In [19]:
test.drop('Item_Weight',axis=1,inplace=True)
test_weight = test.merge(items,how='inner',on='Item_Identifier')
test_weight.isnull().sum()

Item_Identifier                 0
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  1606
Outlet_Location_Type            0
Outlet_Type                     0
Item_Weight                     0
dtype: int64

In [20]:
test_weight

,Item_Identifier,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Weight
0,FDW58,Low Fat,0.007565,Snack Foods,107.8622,OUT049,1999,Medium,Tier 1,Supermarket Type1,20.750
1,FDW58,Low Fat,0.007596,Snack Foods,104.4622,OUT017,2007,NaN,Tier 2,Supermarket Type1,20.750
2,FDW58,Low Fat,0.007584,Snack Foods,107.0622,OUT018,2009,Medium,Tier 3,Supermarket Type2,20.750
3,FDW58,Low Fat,0.000000,Snack Foods,105.9622,OUT046,1997,Small,Tier 1,Supermarket Type1,20.750
4,FDW58,Low Fat,0.007568,Snack Foods,105.8622,OUT045,2002,NaN,Tier 2,Supermarket Type1,20.750
...,...,...,...,...,...,...,...,...,...,...,...
5676,FDN04,Regular,0.014109,Frozen Foods,179.4344,OUT049,1999,Medium,Tier 1,Supermarket Type1,11.800
5677,FDT40,Low Fat,0.095944,Frozen Foods,125.8678,OUT049,1999,Medium,Tier 1,Supermarket Type1,5.985
5678,NCI29,Low Fat,0.032594,Health and Hygiene,140.6154,OUT013,1987,High,Tier 3,Supermarket Type1,8.600
5679,FDP28,Regular,0.080250,Frozen Foods,259.4936,OUT027,1985,Medium,Tier 3,Supermarket Type3,13.650


In [21]:
test_weight['Item_Fat_Content'].replace({
    'low fat': 'Low Fat', 
    'LF': 'Low Fat',
    'reg': 'Regular'
}, inplace = True)

In [22]:
test_weight

,Item_Identifier,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Weight
0,FDW58,Low Fat,0.007565,Snack Foods,107.8622,OUT049,1999,Medium,Tier 1,Supermarket Type1,20.750
1,FDW58,Low Fat,0.007596,Snack Foods,104.4622,OUT017,2007,NaN,Tier 2,Supermarket Type1,20.750
2,FDW58,Low Fat,0.007584,Snack Foods,107.0622,OUT018,2009,Medium,Tier 3,Supermarket Type2,20.750
3,FDW58,Low Fat,0.000000,Snack Foods,105.9622,OUT046,1997,Small,Tier 1,Supermarket Type1,20.750
4,FDW58,Low Fat,0.007568,Snack Foods,105.8622,OUT045,2002,NaN,Tier 2,Supermarket Type1,20.750
...,...,...,...,...,...,...,...,...,...,...,...
5676,FDN04,Regular,0.014109,Frozen Foods,179.4344,OUT049,1999,Medium,Tier 1,Supermarket Type1,11.800
5677,FDT40,Low Fat,0.095944,Frozen Foods,125.8678,OUT049,1999,Medium,Tier 1,Supermarket Type1,5.985
5678,NCI29,Low Fat,0.032594,Health and Hygiene,140.6154,OUT013,1987,High,Tier 3,Supermarket Type1,8.600
5679,FDP28,Regular,0.080250,Frozen Foods,259.4936,OUT027,1985,Medium,Tier 3,Supermarket Type3,13.650


In [23]:
ohe = OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore').set_output(transform='pandas')
SI_F = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')
data_pipe = Pipeline([('SI_F',SI_F),('OHE',ohe)])

In [24]:
transformer = ColumnTransformer([
                                ('PIPE',data_pipe,make_column_selector(dtype_include=object))
                                 ],
                               verbose_feature_names_out=False,remainder='passthrough')

In [25]:
X, y = train_weight.drop('Item_Outlet_Sales', axis = 1), train_weight['Item_Outlet_Sales']

In [26]:
xgbm = XGBRegressor(random_state = 26)
lgbm = LGBMRegressor(random_state = 26,verbose=-1)
lr = LinearRegression()
rf = RandomForestRegressor(random_state=26)
stack = StackingRegressor([('XGB',xgbm),('LGBM',lgbm),('RF',rf)],final_estimator=lr)

In [ ]:
kfolds = KFold(n_splits = 5, shuffle = True, random_state = 26)
# pipe = Pipeline([('OHE',transformer),('XGB',xgbm)])

# params ={
#     'XGB__max_depth': [4, 6,10],
#     'XGB__learning_rate': [0.05, 0.2,0.3],
#     'XGB__n_estimators': [100, 150,200],
# }

pipe_stack = Pipeline([('OHE', transformer), ('stack',stack)])

params = {'stack__XGB__max_depth': [4, 6,10],
          'stack__XGB__learning_rate': [0.05, 0.2,0.3],
          'stack__XGB__n_estimators': [100, 150,200],
         
          'stack__LGBM__max_depth': [4,6,10],
          'stack__LGBM__learning_rate': [0.05, 0.1,0.3],
          'stack__LGBM__n_estimators': [75, 150,200],
          
          'stack__RF__max_depth' : [4,6],
          'stack__RF__n_estimators' : [75,100,150]
         }


rcv = RandomizedSearchCV(estimator= pipe_stack,n_iter=5, cv = kfolds, n_jobs = -1, param_distributions=params, verbose = 2,scoring='neg_root_mean_squared_error')
rcv.fit(X, y)

Fitting 5 folds for each of 5 candidates, totalling 25 fits


In [ ]:
rcv.best_score_,rcv.best_params_

In [ ]:
test

In [ ]:
bm = gcv.best_estimator_
y_pred = bm.predict(test)
y_pred

In [ ]:
submission = pd.read_csv("sample_submission_8RXa3c6.csv")
submission["Item_Outlet_Sales"] = y_pred

In [ ]:
submission.to_csv("AnalyticsVidhyaSubmission.csv", index = False)